In [76]:
import os
import ROOT
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [77]:
f = ROOT.TFile.Open("raw_data/HToBB_000.root")
f.ls()

TFile**		raw_data/HToBB_000.root	/data/hqu/ntuples/delphes/JetClass/Pythia/HOT/train_100M/HToBB_000.root
 TFile*		raw_data/HToBB_000.root	/data/hqu/ntuples/delphes/JetClass/Pythia/HOT/train_100M/HToBB_000.root
  KEY: TTree	tree;1	tree


In [78]:
tree = f.Get("tree")
tree.Print()

******************************************************************************
*Tree    :tree      : tree                                                   *
*Entries :   100000 : Total =       300633281 bytes  File  Size =  161074594 *
*        :          : Tree compression factor =   1.87                       *
******************************************************************************
*Br    0 :part_px   : vector<float,ROOT::Detail::VecOps:                     *
*         | :RAdoptAllocator<float> >                                        *
*Entries :   100000 : Total  Size=   18252571 bytes  File Size  =   17737496 *
*Baskets :      511 : Basket Size=      32000 bytes  Compression=   1.03     *
*............................................................................*
*Br    1 :part_py   : vector<float,ROOT::Detail::VecOps:                     *
*         | :RAdoptAllocator<float> >                                        *
*Entries :   100000 : Total  Size=   18252571 bytes 

In [79]:
folder = "raw_data/"
output_folder = "ready_data/"

for filename in os.listdir(folder):
    file_path = os.path.join(folder, filename)
    df = ROOT.RDataFrame("tree", file_path)
    
    total_entries = df.Count().GetValue()
    split_index = int(total_entries * 0.7)
    
    train_df = df.Range(0, split_index)
    val_df = df.Range(split_index, total_entries)

    print(f"file: {filename}\n" \
              f"entries: {total_entries}\n" \
              f"train: {train_df.Count().GetValue()}\n" \
              f"eval: {val_df.Count().GetValue()}\n")

    train_output = os.path.join(output_folder, f"{filename.replace('.root', '')}_train.root")
    val_output = os.path.join(output_folder, f"{filename.replace('.root', '')}_val.root")

    train_df.Snapshot("tree", train_output)
    val_df.Snapshot("tree", val_output)

file: HToBB_000.root
entries: 100000
train: 70000
eval: 30000

file: HToGG_000.root
entries: 100000
train: 70000
eval: 30000



In [80]:
columns = df.GetColumnNames()
for col in sorted(columns):
    print(col)

aux_genpart_eta
aux_genpart_phi
aux_genpart_pid
aux_genpart_pt
aux_truth_match
jet_energy
jet_eta
jet_nparticles
jet_phi
jet_pt
jet_sdmass
jet_tau1
jet_tau2
jet_tau3
jet_tau4
label_H4q
label_Hbb
label_Hcc
label_Hgg
label_Hqql
label_QCD
label_Tbl
label_Tbqq
label_Wqq
label_Zqq
part_charge
part_d0err
part_d0val
part_deta
part_dphi
part_dzerr
part_dzval
part_energy
part_isChargedHadron
part_isElectron
part_isMuon
part_isNeutralHadron
part_isPhoton
part_px
part_py
part_pz


In [81]:
jet_features = [
    "jet_energy", "jet_eta", "jet_nparticles", "jet_phi", 
    "jet_pt", "jet_sdmass", "jet_tau1", "jet_tau2", "jet_tau3", "jet_tau4"
]
labels = ["label_Hbb", "label_Hgg"]
selected_columns = jet_features + labels

folder = "ready_data/"

for filename in os.listdir(folder):
    if filename.startswith("."):
        continue
        
    file_path = os.path.join(folder, filename)
    size_before = os.path.getsize(os.path.join("raw_data/", filename)) / (1024 * 1024)
    
    temp_path = file_path + ".tmp"
    ROOT.RDataFrame("tree", file_path).Snapshot("tree", temp_path, selected_columns)
    os.replace(temp_path, file_path)

    size_after = os.path.getsize(file_path) / (1024 * 1024)
    print(f"{size_before:.2f} MB -> {size_after:.2f} MB")

153.62 MB -> 3.15 MB
200.32 MB -> 3.12 MB
